In [ ]:
from src.trainer import Trainer
from loguru import logger
import sys
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stderr, level="WARNING")

In [ ]:
rename = {
    "sentence_GROUP": "Quest./Decl.",
    "subj_TYPE": "Subj. type",
    "sentence_TENSE": "Tense",
    "subj_PERS": "Subj. pers.",
    "subj_ANIM": "Subj. anim.",
    "subj_type": "Subj. type",
    "verb_ZIPF": "Verb freq.",
    "prep_LEMMA": "Preposition",
    "verb_type": "Trans./Intrans.",
    "has_main": "Quest./Decl.",
    "sentence_length": "Length",
    "has_embed": "Prop. attitude",
    "sentence_RC_attached": "Attach. site",
    "subj_NUM": "Subj. num.",
    "subj_ZIPF": "Subj. freq.",
    "verb_LEMMA": "Verb lemma",
    "sentence_CLAUSE": "RC type",
    "sentence_PP_attached": "Attach. site",
    "subj_GEN": "Subj. gender",
    "obj_ZIPF": "Obj. freq.",
    "sv": "Decl.",
    "question": "Quest.",
    "sg": "Singular",
    "pl": "Plural",
    "peripheral": "Periph.",
    "center_embedding": "Center embed.",
    "obj_NUM": "Obj. num.",
    "obj_GEN": "Obj. gender",
    "embed_NUM": "Embed. num.",
    "embed_GEN": "Embed. gender",
    "embed_ZIPF": "Embed. freq.",
    "intervener_NUM": "Interv. num.",
    "intervener_GEN": "Interv. gender",
    "embedobj_NUM": "Embed. num.",
    "embedobj_GEN": "Embed. gender",
    "embedobj_ZIPF": "Embed. freq.",
    "verb_PERS": "Verb pers.",
    "verb_NUM": "Verb num.",
    "verb_finite": "Finite verb",
    "has_relative_clause": "RC",
    "has_objrel": "Obj. rel.",
    "has_subjrel": "Subj. rel.",
    "incongruence_poss_subj_count": "#Mismatch poss. subj.",
    "incongruence_poss_obj_count": "#Mismatch poss. obj.",
    "incongruence_subj_obj_count": "#Mismatch subj. obj.",
    "incongruence_subj_embedsubj_count": "#Mismatch subj. embed.",
    "rel_type": "Rel. type",
    "clause_type": "Clause type",
    "bound_variable": "Bound var.",
    "mean_zipf": "Mean freq.",
    "long_range_agreement_with_attractor": "LR agree.",
    "short_sentence": "Short-Sentence",
    "short_sentence_2w": "Short-Sentence",
    "relative_clause": "Relative-Clause",
    "long_range_agreement": "Long-Range-Agreement",
    "large": "Large",
    # "layer": "Layer",
    # "importance": "Feature Importance"
}

In [ ]:
importances = []
spearman = []
run_id = logger.add("run.log", level="WARNING")
datasets = [
    "short_sentence",
    "relative_clause",
    "long_range_agreement",
    "large",
]
weights = {}
for dataset in datasets:
    for layer in [5]:
        trainer = Trainer(
            dataframe={"csv_path": f"datasets/{dataset}.csv"},
            representations={
                "level": "sentence",
                "layer": layer,
            },
        )
        state_dict, _ = trainer.train()
        model, _ = trainer.init(state_dict=state_dict)
        W = model.get_formatted_W()
        renamed_features = [rename.get(f, f) for f in trainer.features]
        W.columns = renamed_features
        W.index = renamed_features
        weights[dataset] = W

In [ ]:
weights[dataset] = W
thresh = np.nanquantile(W.abs().values, 0.9)
annot = W.copy()
annot = annot.round(2)
annot = annot.astype(str)
annot[W.abs() <= thresh] = ""
sns.heatmap(
    W,
    cmap="RdBu",
    center=0,
    annot=annot,
    fmt="",
    square=True,
    cbar_kws={"shrink": 0.8, "label": "Weight"},
)

In [ ]:
all_weights = np.concatenate(
    [weights[ds].values.flatten() for ds in datasets if weights[ds].size > 0]
)
valid_weights = all_weights[~np.isnan(all_weights)]

vmin = np.nanmin(valid_weights)
vmax = np.nanmax(valid_weights)
thresh = 0.11

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes_flat = axes.flatten()
cbar_ax = fig.add_axes([0.3, 0, 0.4, 0.01])

num_plots = min(len(datasets), 4)

for i in range(num_plots):
    dataset = datasets[i]
    ax = axes_flat[i]
    W = weights[dataset]

    annot = W.round(2 if dataset != "large" else 1).astype(str)
    mask = np.isnan(W) | (np.abs(W) <= thresh)
    annot[mask] = ""

    sns.heatmap(
        W,
        cmap="RdBu",
        center=0,
        annot=annot,
        fmt="",
        square=True,
        cbar=i == 0,
        cbar_ax=cbar_ax if i == 0 else None,
        cbar_kws={"label": "Weight", "orientation": "horizontal"},
        ax=ax,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"{rename.get(dataset, dataset)}")
    last_ax = ax
fig.suptitle("BERT layer 5", fontsize=16, y=1.025)
fig.tight_layout(rect=[0, 0, 0.9, 1.025], pad=0)
plt.savefig(".figs/spd_weights.pdf", bbox_inches="tight")